# Notes

В профиле на платформе fatsecret надо в вайтлист айпиадресов внести айпи своего приложения

# Import

In [1]:
import os
import webbrowser
from getpass import getpass
from urllib.parse import parse_qs
from dotenv import load_dotenv
from requests_oauthlib import OAuth1Session
from datetime import date
import pandas as pd

c:\Users\Илья\Desktop\Саморазвитие\Бизнес (папка резервирования)\venv\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.0) or chardet (7.4.3)/charset_normalizer (3.4.2) doesn't match a supported version!
  warnings.warn(


# Аутентификация

In [2]:
load_dotenv()
CLIENT_ID = os.getenv('ConsumerKey')
CLIENT_SECRET = os.getenv('ConsumerSecret')

def parse_token(text):
    data = parse_qs(text)
    return (data.get('oauth_token', [None])[0],
            data.get('oauth_token_secret', [None])[0])

session = OAuth1Session(CLIENT_ID, CLIENT_SECRET, callback_uri='oob', signature_type='BODY')
resp = session.post('https://authentication.fatsecret.com/oauth/request_token')
resp.raise_for_status()
request_token, request_token_secret = parse_token(resp.text)

auth_url = f'https://authentication.fatsecret.com/oauth/authorize?oauth_token={request_token}'
print(f'Перейдите по ссылке и разрешите доступ:\n{auth_url}')
webbrowser.open(auth_url)

verifier = getpass('Введите PIN-код: ').strip()

access_session = OAuth1Session(
    CLIENT_ID, CLIENT_SECRET,
    resource_owner_key=request_token,
    resource_owner_secret=request_token_secret,
    verifier=verifier,
    signature_type='BODY'
)
resp = access_session.post('https://authentication.fatsecret.com/oauth/access_token')
resp.raise_for_status()
access_token, access_token_secret = parse_token(resp.text)

api_session = OAuth1Session(
    CLIENT_ID, CLIENT_SECRET,
    resource_owner_key=access_token,
    resource_owner_secret=access_token_secret
)


Перейдите по ссылке и разрешите доступ:
https://authentication.fatsecret.com/oauth/authorize?oauth_token=ab5c6ac820ab4a93a5661ae8d9ac6d03


# Данные

## Пользователь

In [4]:
response = api_session.get(
    'https://platform.fatsecret.com/rest/server.api',
    params={'method': 'profile.get', 'format': 'json'}
)
response.raise_for_status()

In [8]:
response.json()

{'profile': {'goal_weight_kg': '92.0000',
  'height_cm': '185.00',
  'height_measure': 'Cm',
  'last_weight_date_int': '20507',
  'last_weight_kg': '87.5000',
  'weight_measure': 'Kg'}}

## Дневник питания

In [10]:
def date_to_int(d):
    epoch = date(1970, 1, 1)
    return (d - epoch).days

start_date = date_to_int(date(2026, 4, 1))


response = api_session.get(
    'https://platform.fatsecret.com/rest/server.api',
    params={'method': 'food_entries.get.v2', 
            'format': 'json',
            'date': start_date}
)
response.raise_for_status()

In [11]:
pd.DataFrame(response.json()['food_entries']['food_entry'])

,calories,carbohydrate,cholesterol,date_int,fat,fiber,food_entry_description,food_entry_id,food_entry_name,food_id,meal,monounsaturated_fat,number_of_units,polyunsaturated_fat,potassium,protein,saturated_fat,serving_id,sodium,sugar
0,250,55.93,0,20544,0.72,13.5,301 g Булгур (Вареный),23079810677,Булгур (Вареный),39693,Breakfast,0.093,301.000,0.295,205,9.27,0.126,62424,15,0.30
1,167,10.50,NaN,20544,5.00,NaN,2 1/2 servings Савушкин Продукт Йогурт Греческ...,23077583829,Савушкин Продукт Йогурт Греческий Teos 2%,17400944,Breakfast,NaN,2.500,NaN,NaN,20.00,NaN,16411429,NaN,19.50
2,540,91.80,NaN,20544,17.55,NaN,1.35 servings Красный Пищевик Зефир в Шоколаде,23077490604,Красный Пищевик Зефир в Шоколаде,11083037,Breakfast,NaN,1.350,NaN,NaN,4.05,NaN,10548954,NaN,NaN
3,122,27.74,NaN,20544,0,NaN,0.38 serving Красный Пищевик Мармелад,23077490603,Красный Пищевик Мармелад,78571731,Breakfast,NaN,0.380,NaN,NaN,2.85,NaN,63621977,NaN,NaN
4,617,10.89,NaN,20544,36.30,NaN,3 2/3 servings Петелинка Шницель из Куриной Гр...,23083560323,Петелинка Шницель из Куриной Грудки,81764479,Lunch,NaN,3.630,NaN,NaN,61.71,NaN,65982567,NaN,NaN
5,239,6.58,85,20544,3.19,NaN,1.879 servings Ultra Whey Pro,23083560257,Ultra Whey Pro,5686909,Lunch,NaN,1.879,NaN,282,45.28,1.880,5522529,216,5.64
6,45,4.79,NaN,20544,1.53,NaN,"1.02 servings Молоко 1,5%",23083548897,"Молоко 1,5%",17052724,Lunch,NaN,1.020,NaN,NaN,3.06,NaN,16088703,102,NaN
7,358,89.60,NaN,20544,0,NaN,1.12 servings Маркет Перекресток Мармелад со В...,23087055648,Маркет Перекресток Мармелад со Вкусом Дыни,83427867,Other,NaN,1.120,NaN,NaN,0,NaN,67252390,NaN,NaN


## Наиболее употребляемая

In [13]:
response = api_session.get(
    'https://platform.fatsecret.com/rest/server.api',
    params={'method': 'foods.get_most_eaten.v2', 
            'format': 'json'}
)
response.raise_for_status()
response.json()

{'foods': {'food': [{'brand_name': 'Савушкин Продукт',
    'food_id': '17400944',
    'food_name': 'Йогурт Греческий Teos 2%',
    'food_type': 'Brand',
    'food_url': 'https://foods.fatsecret.com/calories-nutrition/ru/%D0%A1%D0%B0%D0%B2%D1%83%D1%88%D0%BA%D0%B8%D0%BD-%D0%9F%D1%80%D0%BE%D0%B4%D1%83%D0%BA%D1%82/%D0%99%D0%BE%D0%B3%D1%83%D1%80%D1%82-%D0%93%D1%80%D0%B5%D1%87%D0%B5%D1%81%D0%BA%D0%B8%D0%B9-teos-2/100%D0%B3',
    'number_of_units': '2.500',
    'serving_id': '16411429'},
   {'brand_name': 'Просто Молоко',
    'food_id': '17052724',
    'food_name': 'Молоко 1,5%',
    'food_type': 'Brand',
    'food_url': 'https://foods.fatsecret.com/calories-nutrition/ru/%D0%9F%D1%80%D0%BE%D1%81%D1%82%D0%BE-%D0%9C%D0%BE%D0%BB%D0%BE%D0%BA%D0%BE/%D0%9C%D0%BE%D0%BB%D0%BE%D0%BA%D0%BE-15/100%D0%BC%D0%BB',
    'number_of_units': '2.190',
    'serving_id': '16088703'},
   {'food_id': '39693',
    'food_name': 'Bulgur (Cooked)',
    'food_type': 'Generic',
    'food_url': 'https://foods.fatsecret.com

## Недавно употребленная

In [14]:
response = api_session.get(
    'https://platform.fatsecret.com/rest/server.api',
    params={'method': 'foods.get_recently_eaten.v2', 
            'format': 'json'}
)
response.raise_for_status()
response.json()

{'foods': {'food': [{'brand_name': 'Маркет Перекресток',
    'food_id': '83427867',
    'food_name': 'Мармелад со Вкусом Дыни',
    'food_type': 'Brand',
    'food_url': 'https://foods.fatsecret.com/calories-nutrition/ru/%D0%9C%D0%B0%D1%80%D0%BA%D0%B5%D1%82-%D0%9F%D0%B5%D1%80%D0%B5%D0%BA%D1%80%D0%B5%D1%81%D1%82%D0%BE%D0%BA/%D0%9C%D0%B0%D1%80%D0%BC%D0%B5%D0%BB%D0%B0%D0%B4-%D1%81%D0%BE-%D0%92%D0%BA%D1%83%D1%81%D0%BE%D0%BC-%D0%94%D1%8B%D0%BD%D0%B8/100%D0%B3',
    'number_of_units': '1.280',
    'serving_id': '67252390'},
   {'brand_name': 'Universal Nutrition',
    'food_id': '5686909',
    'food_name': 'Ultra Whey Pro',
    'food_type': 'Brand',
    'food_url': 'https://foods.fatsecret.com/calories-nutrition/ru/universal-nutrition/ultra-whey-pro/1-%D0%BF%D0%BE%D1%80%D1%86%D0%B8%D1%8F',
    'number_of_units': '1.667',
    'serving_id': '5522529'},
   {'brand_name': 'Савушкин Продукт',
    'food_id': '17400944',
    'food_name': 'Йогурт Греческий Teos 2%',
    'food_type': 'Brand',
    'fo

##  Сохраненные блюда

In [16]:
response = api_session.get(
    'https://platform.fatsecret.com/rest/server.api',
    params={'method': 'saved_meals.get.v2', 
            'format': 'json'}
)
response.raise_for_status()
response.json()

{'saved_meals': {'saved_meal': [{'meals': 'Breakfast,Lunch,Dinner,Other',
    'saved_meal_description': None,
    'saved_meal_id': '46183002',
    'saved_meal_name': 'Банан'},
   {'meals': 'Breakfast,Lunch,Dinner,Other',
    'saved_meal_description': None,
    'saved_meal_id': '46076370',
    'saved_meal_name': 'Булгур'},
   {'meals': 'Breakfast,Lunch,Dinner,Other',
    'saved_meal_description': None,
    'saved_meal_id': '47395258',
    'saved_meal_name': 'Запеканка (Мила)'},
   {'meals': 'Breakfast,Lunch,Dinner,Other',
    'saved_meal_description': None,
    'saved_meal_id': '45849824',
    'saved_meal_name': 'Йогурт Теос'},
   {'meals': 'Breakfast,Lunch,Dinner,Other',
    'saved_meal_description': None,
    'saved_meal_id': '45867564',
    'saved_meal_name': 'Кетчуп'},
   {'meals': 'Breakfast,Lunch,Dinner,Other',
    'saved_meal_description': None,
    'saved_meal_id': '46519088',
    'saved_meal_name': 'Кофе'},
   {'meals': 'Breakfast,Lunch,Dinner,Other',
    'saved_meal_descriptio

## Упражнения константы

In [17]:
response = api_session.get(
    'https://platform.fatsecret.com/rest/server.api',
    params={'method': 'exercises.get.v2', 
            'format': 'json'}
)
response.raise_for_status()
response.json()

{'exercises': {'exercise': [{'exercise_id': '0', 'exercise_name': 'Other'},
   {'exercise_id': '77', 'exercise_name': 'Abdominal (Sit Ups)'},
   {'exercise_id': '135', 'exercise_name': 'Aerobics'},
   {'exercise_id': '144', 'exercise_name': 'Air Climber'},
   {'exercise_id': '137', 'exercise_name': 'Arc Trainer'},
   {'exercise_id': '149', 'exercise_name': 'Archery'},
   {'exercise_id': '3', 'exercise_name': 'Badminton'},
   {'exercise_id': '124', 'exercise_name': 'Ballet'},
   {'exercise_id': '174', 'exercise_name': 'Baseball'},
   {'exercise_id': '4', 'exercise_name': 'Basketball'},
   {'exercise_id': '125', 'exercise_name': 'Bathing'},
   {'exercise_id': '5', 'exercise_name': 'Fast Bicycling'},
   {'exercise_id': '6', 'exercise_name': 'Lesurely Bicycling'},
   {'exercise_id': '7', 'exercise_name': 'Moderate Bicycling'},
   {'exercise_id': '8', 'exercise_name': 'Slow Bicycling'},
   {'exercise_id': '9', 'exercise_name': 'Very Fast Bicycling'},
   {'exercise_id': '73', 'exercise_name'

## Упражнения за дату + калории

In [19]:
response = api_session.get(
    'https://platform.fatsecret.com/rest/server.api',
    params={'method': 'exercise_entries.get.v2', 
            'format': 'json',
            'date':start_date}
)
response.raise_for_status()
response.json()

{'exercise_entries': {'exercise_entry': [{'calories': '1470',
    'exercise_id': '2',
    'exercise_name': 'Resting',
    'is_template_value': '1',
    'minutes': '960'},
   {'calories': '662',
    'exercise_id': '1',
    'exercise_name': 'Sleeping',
    'is_template_value': '1',
    'minutes': '480'}]}}